In [ ]:
import re
import pandas as pd
from datetime import datetime

df = pd.read_csv("bot_messages.csv")
recap_mask = ["🏁 • 🆚" in x and "@" not in x and "🤖" not in x for x in df["text"]]
df_text = df[recap_mask]['text'][::-1]

In [127]:
def extract_matches(text: str, previous_date: str | None) -> list[dict]:
    if not text:
        return []

    text = str(text)

    date_match = re.search(r"Today's Matches \((\d{2}-\d{2}-\d{4})\)", text)
    match_date = date_match.group(1) if date_match else previous_date

    rows = []

    # Divide il testo in blocchi, uno per match
    blocks = re.split(r'(?=🏁\s*•\s*🆚)', text)

    for block in blocks:
        block = block.strip()

        # Salta tutto ciò che non è un blocco match valido
        if not block.startswith("🏁"):
            continue

        teams_match = re.search(r"🏁\s*•\s*🆚\s*(.*?)\s*-\s*(.*?)\n", block)
        time_match = re.search(r"🕒\s*Time:\s*(.*?)\n", block)
        league_match = re.search(r"🏆\s*League:\s*(.*?)\n", block)
        match_id_match = re.search(r"🆔\s*Match ID:\s*(\S+)", block)

        strategies_match = re.search(
            r"[🧩🧠]\s*Strategies:\s*\n(.*?)(?=\n\s*🆔\s*Match ID:|\Z)",
            block,
            re.S
        )

        # Se il blocco è incompleto, skippalo
        if not all([teams_match, time_match, league_match, strategies_match, match_id_match]):
            continue

        home_team = teams_match.group(1).strip()
        away_team = teams_match.group(2).strip()
        time = time_match.group(1).strip()
        league = league_match.group(1).strip()
        match_id = match_id_match.group(1).strip()
        strategies_block = strategies_match.group(1)

        for line in strategies_block.splitlines():
            line = line.strip()
            if not line:
                continue

            m = re.match(r"([✅❌❓])\s*(.*?)\s*\(([+-]?\d+(?:\.\d+)?)\)\s*$", line)
            if not m:
                continue

            icon, strategy, ret = m.groups()

            # Skippa strategie con ❓
            if icon == "❓":
                continue

            rows.append({
                "date": match_date,
                "match_id": match_id,
                "time": time,
                "league": league,
                "home_team": home_team,
                "away_team": away_team,
                "strategy": strategy.strip(),
                "result": True if icon == "✅" else False,
                "return": float(ret),
            })

    return rows

In [209]:
tot_matches = list()
previous_date = None
for row in tqdm(df_text.values):
    item = extract_matches(row, previous_date=previous_date)
    for d in item:
        if d["date"]:
            previous_date = d["date"]
    tot_matches += item

df_final = pd.DataFrame(tot_matches)
df_final = df_final.set_index("match_id", drop=True)
df_final["date"] = [datetime.strptime(x, "%d-%m-%Y") for x in df_final["date"]]
df_final["week"] = df_final["date"].dt.to_period("W")

df_final = df_final.sort_values("date") 

monday_start = df_final["date"].min() - pd.to_timedelta(df_final["date"].min().weekday(), unit="D")
df_final["week"] = ((df_final["date"] - monday_start).dt.days // 7) + 1

100%|██████████| 73/73 [00:00<00:00, 15220.93it/s]


In [223]:
df_final.groupby("week")["return"].sum()

week
1      3.20
2      2.35
3      0.35
4     -2.30
5      6.00
6      7.05
7     27.88
8      4.90
9      1.35
10     3.95
Name: return, dtype: float64

In [179]:
3*2.98

8.94

In [227]:
min(df_final["date"])

Timestamp('2026-01-07 00:00:00')

In [202]:
strategy_dict = dict()
df_final = df_final[df_final["week"] != 7]
for strategy in df_final['strategy'].unique():
    gain = df_final[df_final['strategy'] == strategy].groupby("week")["return"].sum().mean()
    # if gain > 10:
    #     continue
    strategy_dict = strategy_dict | {strategy: gain}

strategy_df = pd.DataFrame(strategy_dict, index=["gain"]).T.sort_values(by='gain', ascending=False)

strategy_df

,gain
Exchange,2.250000
Win Goal,1.000000
DNB 1,0.987500
Win x2 HT,0.760000
Over 0.5 HT (Live),0.733333
Over 1.5 (M2-X),0.700000
Draw Win 1X,0.600000
Over 1.5 (M-2X),0.500000
Multigoal 1-3 Home,0.420000
Goal HT,0.388889


In [208]:
df_final[df_final['strategy'] == "DNB 1"]

,date,time,league,home_team,away_team,strategy,result,return,week
match_id,,,,,,,,,
d19509254,2026-01-16,20:30,Serie B,Sampdoria,Virtus Entella,DNB 1,True,0.3,2
d19439443,2026-01-16,21:00,La Liga,Espanyol,Girona,DNB 1,False,-1.0,2
d19429865,2026-01-16,20:00,Olanda B,VVV,Venlo - Jong AZ,DNB 1,False,-1.0,2
d19577439,2026-01-16,11:45,Australia,Perth Glory,Brisbane Roar,DNB 1,False,-1.0,2
d19432620,2026-01-17,16:00,League One,Peterborough United,Plymouth Argyle,DNB 1,False,-1.0,2
...,...,...,...,...,...,...,...,...,...
d19503905,2026-03-15,17:30,Serie C: Girone C,Casertana,Monopoli,DNB 1,True,0.4,10
d19439529,2026-03-15,21:00,La Liga,Real Sociedad,Osasuna,DNB 1,True,0.4,10
d19622023,2026-03-15,20:00,Brasile,Internacional,Bahia,DNB 1,False,-1.0,10


In [170]:
df_final[df_final['strategy'] == "BTTS"]['result'].sum() / df_final[df_final['strategy'] == "BTTS"].shape[0]

np.float64(0.6833333333333333)